# <b>UFC fight prediction model </b>

# Import dependencies

In [2]:
import pandas as pd
import sqlite3
import sklearn

# Bring in data

In [3]:
"""
winner_id (label) -> FK for fighter_details.csv
fight_id -> FK for fight_details.csv
"""
Outcomes = pd.read_csv("data/raw/event_details.csv")
Outcomes.head(5)




,event_id,fight_id,date,location,winner,winner_id
0,6e380a4d73ab4f0e,5bcd8abf6da01dae,"September 06, 2025","Paris, Ile-de-France, France",Nassourdine Imavov,881bf86d4cba8578
1,6e380a4d73ab4f0e,d14fea43712707f0,"September 06, 2025","Paris, Ile-de-France, France",Benoit Saint Denis,c2299ec916bc7c56
2,6e380a4d73ab4f0e,0149e875b31e57f8,"September 06, 2025","Paris, Ile-de-France, France",Modestas Bukauskas,476fe566d2df676e
3,6e380a4d73ab4f0e,69b31593175f77ec,"September 06, 2025","Paris, Ile-de-France, France",Mason Jones,f6ad6a1e4d600e0d
4,6e380a4d73ab4f0e,a47123ae6e048f11,"September 06, 2025","Paris, Ile-de-France, France",Axel Sola,1d2d98ddd05b38fb


In [4]:
Fighters = pd.read_csv("data/raw/fighter_details.csv")
Fighters.head(5)

,id,name,nick_name,wins,losses,draws,height,weight,reach,stance,dob,splm,str_acc,sapm,str_def,td_avg,td_avg_acc,td_def,sub_avg
0,8f382b3baa954d2a,Jessica Aguilar,Jag,20,8,0,160.02,52.16,160.02,Orthodox,"May 08, 1982",4.93,50,7.19,53,0.94,25,50,0.2
1,483a953b18d73bb3,Deron Winn,NaN,7,3,0,167.64,83.91,177.80,Orthodox,"Jun 13, 1989",4.55,44,6.21,46,4.28,52,40,0.0
2,232c582f29f8f65e,Gegard Mousasi,NaN,42,6,2,187.96,83.91,193.04,Orthodox,"Aug 01, 1985",3.75,50,1.21,68,1.59,60,59,1.1
3,236a37d96d476164,Mike Pierce,NaN,17,7,0,172.72,77.11,180.34,Orthodox,"Sep 01, 1980",2.62,42,2.36,62,3.08,42,71,0.2
4,203c957eac95dd87,Hyun Gyu Lim,The Ace,13,7,1,190.50,77.11,195.58,Orthodox,"Jan 16, 1985",4.45,41,5.07,51,0.23,100,73,0.2


## drop non-numeric columns in fighter table (besides name)

In [5]:
# drop non-numeric columns
Fighters = Fighters.drop(["name", "nick_name", "dob", "stance"], axis=1)
Fighters.head()




,id,wins,losses,draws,height,weight,reach,splm,str_acc,sapm,str_def,td_avg,td_avg_acc,td_def,sub_avg
0,8f382b3baa954d2a,20,8,0,160.02,52.16,160.02,4.93,50,7.19,53,0.94,25,50,0.2
1,483a953b18d73bb3,7,3,0,167.64,83.91,177.80,4.55,44,6.21,46,4.28,52,40,0.0
2,232c582f29f8f65e,42,6,2,187.96,83.91,193.04,3.75,50,1.21,68,1.59,60,59,1.1
3,236a37d96d476164,17,7,0,172.72,77.11,180.34,2.62,42,2.36,62,3.08,42,71,0.2
4,203c957eac95dd87,13,7,1,190.50,77.11,195.58,4.45,41,5.07,51,0.23,100,73,0.2


In [6]:
Fights = pd.read_csv("data/raw/fight_details.csv")
Fights.head(5)

,event_name,event_id,fight_id,r_name,r_id,b_name,b_id,division,title_fight,method,...,b_clinch_acc,b_ground_landed,b_ground_atmpted,b_ground_acc,b_landed_head_per,b_landed_body_per,b_landed_leg_per,b_landed_dist_per,b_landed_clinch_per,b_landed_ground_per
0,UFC Fight Night: Imavov vs. Borralho,6e380a4d73ab4f0e,d14fea43712707f0,Benoit Saint Denis,c2299ec916bc7c56,Mauricio Ruffy,9c393e836a852f30,lightweight,0,Submission,...,NaN,0.0,0.0,NaN,80.0,20.0,0.0,100.0,0.0,0.0
1,UFC Fight Night: Imavov vs. Borralho,6e380a4d73ab4f0e,5bcd8abf6da01dae,Nassourdine Imavov,881bf86d4cba8578,Caio Borralho,4126a78111c0855a,middleweight,0,Decision - Unanimous,...,100.0,0.0,0.0,NaN,59.0,21.0,19.0,89.0,10.0,0.0
2,UFC Fight Night: Imavov vs. Borralho,6e380a4d73ab4f0e,0149e875b31e57f8,Modestas Bukauskas,476fe566d2df676e,Paul Craig,eabf206b162b3b83,light heavyweight,0,KO/TKO,...,100.0,0.0,0.0,NaN,0.0,33.0,66.0,83.0,16.0,0.0
3,UFC Fight Night: Imavov vs. Borralho,6e380a4d73ab4f0e,69b31593175f77ec,Bolaji Oki,4bdedbdeedff7d1d,Mason Jones,f6ad6a1e4d600e0d,lightweight,0,KO/TKO,...,67.0,32.0,49.0,65.0,91.0,3.0,5.0,40.0,3.0,56.0
4,UFC Fight Night: Imavov vs. Borralho,6e380a4d73ab4f0e,a47123ae6e048f11,Axel Sola,1d2d98ddd05b38fb,Rhys McKee,f748267c4ab6c127,welterweight,0,KO/TKO,...,53.0,0.0,0.0,NaN,81.0,18.0,0.0,69.0,30.0,0.0


# try to make new table of relevant info

In [7]:
RelInfo = Fights[["event_id", "fight_id", "r_id", "r_name", "b_id", "b_name"]]
RelInfo.head(5)
# # add r fighters

RFighters = Fighters.add_prefix("r_")

RelInfo = RelInfo.merge(RFighters, left_on="r_id", right_on="r_id")

# # add b fighters
BFighters = Fighters.add_prefix("b_")
RelInfo = RelInfo.merge(BFighters, left_on="b_id", right_on="b_id")
RelInfo.head(5)


,event_id,fight_id,r_id,r_name,b_id,b_name,r_wins,r_losses,r_draws,r_height,...,b_weight,b_reach,b_splm,b_str_acc,b_sapm,b_str_def,b_td_avg,b_td_avg_acc,b_td_def,b_sub_avg
0,6e380a4d73ab4f0e,d14fea43712707f0,c2299ec916bc7c56,Benoit Saint Denis,9c393e836a852f30,Mauricio Ruffy,15,3,0,180.34,...,70.31,190.5,3.84,57,4.16,60,0.00,0,84,0.0
1,ca936c67687789e9,231968c02cdfe65e,887961364be5ceb3,King Green,9c393e836a852f30,Mauricio Ruffy,32,17,1,177.80,...,70.31,190.5,3.84,57,4.16,60,0.00,0,84,0.0
2,118463dd8db16e7f,c7190cd03e53fb61,c2299ec916bc7c56,Benoit Saint Denis,873626e5547b5235,Kyle Prepolec,15,3,0,180.34,...,70.31,177.8,2.87,35,5.19,49,0.00,0,40,0.0
3,351264d11286d09a,003c84fc7c3fa028,49858bf46dabf6fb,Nordine Taleb,873626e5547b5235,Kyle Prepolec,15,7,0,185.42,...,70.31,177.8,2.87,35,5.19,49,0.00,0,40,0.0
4,ef61d9f5176b3200,dfa8fd0b82b37fa1,c2299ec916bc7c56,Benoit Saint Denis,d945aae53e3e54e6,Thiago Moises,15,3,0,180.34,...,70.31,177.8,2.48,42,4.08,53,1.31,35,52,1.1


## add in label (fight winner)

In [8]:
# it's in order of newest to oldest date. To avoid using
Outcomes = Outcomes[["fight_id", "winner_id", "date"]]
RelInfo = RelInfo.merge(Outcomes, on="fight_id")
RelInfo.head()


,event_id,fight_id,r_id,r_name,b_id,b_name,r_wins,r_losses,r_draws,r_height,...,b_splm,b_str_acc,b_sapm,b_str_def,b_td_avg,b_td_avg_acc,b_td_def,b_sub_avg,winner_id,date
0,6e380a4d73ab4f0e,d14fea43712707f0,c2299ec916bc7c56,Benoit Saint Denis,9c393e836a852f30,Mauricio Ruffy,15,3,0,180.34,...,3.84,57,4.16,60,0.00,0,84,0.0,c2299ec916bc7c56,"September 06, 2025"
1,ca936c67687789e9,231968c02cdfe65e,887961364be5ceb3,King Green,9c393e836a852f30,Mauricio Ruffy,32,17,1,177.80,...,3.84,57,4.16,60,0.00,0,84,0.0,9c393e836a852f30,"March 08, 2025"
2,118463dd8db16e7f,c7190cd03e53fb61,c2299ec916bc7c56,Benoit Saint Denis,873626e5547b5235,Kyle Prepolec,15,3,0,180.34,...,2.87,35,5.19,49,0.00,0,40,0.0,c2299ec916bc7c56,"May 10, 2025"
3,351264d11286d09a,003c84fc7c3fa028,49858bf46dabf6fb,Nordine Taleb,873626e5547b5235,Kyle Prepolec,15,7,0,185.42,...,2.87,35,5.19,49,0.00,0,40,0.0,49858bf46dabf6fb,"May 04, 2019"
4,ef61d9f5176b3200,dfa8fd0b82b37fa1,c2299ec916bc7c56,Benoit Saint Denis,d945aae53e3e54e6,Thiago Moises,15,3,0,180.34,...,2.48,42,4.08,53,1.31,35,52,1.1,c2299ec916bc7c56,"September 02, 2023"


### R-fighter wins => winner_id = 1
### b-fighter wins => winner_id = 0

In [9]:
def encode_winner(df):
    encoded = []
    for _, row in df.iterrows():
        encoded.append(int(row["winner_id"] == row["r_id"]))

    df["winner_id"] = encoded
    return df

RelCopy = encode_winner(RelInfo)
RelCopy.head()



,event_id,fight_id,r_id,r_name,b_id,b_name,r_wins,r_losses,r_draws,r_height,...,b_splm,b_str_acc,b_sapm,b_str_def,b_td_avg,b_td_avg_acc,b_td_def,b_sub_avg,winner_id,date
0,6e380a4d73ab4f0e,d14fea43712707f0,c2299ec916bc7c56,Benoit Saint Denis,9c393e836a852f30,Mauricio Ruffy,15,3,0,180.34,...,3.84,57,4.16,60,0.00,0,84,0.0,1,"September 06, 2025"
1,ca936c67687789e9,231968c02cdfe65e,887961364be5ceb3,King Green,9c393e836a852f30,Mauricio Ruffy,32,17,1,177.80,...,3.84,57,4.16,60,0.00,0,84,0.0,0,"March 08, 2025"
2,118463dd8db16e7f,c7190cd03e53fb61,c2299ec916bc7c56,Benoit Saint Denis,873626e5547b5235,Kyle Prepolec,15,3,0,180.34,...,2.87,35,5.19,49,0.00,0,40,0.0,1,"May 10, 2025"
3,351264d11286d09a,003c84fc7c3fa028,49858bf46dabf6fb,Nordine Taleb,873626e5547b5235,Kyle Prepolec,15,7,0,185.42,...,2.87,35,5.19,49,0.00,0,40,0.0,1,"May 04, 2019"
4,ef61d9f5176b3200,dfa8fd0b82b37fa1,c2299ec916bc7c56,Benoit Saint Denis,d945aae53e3e54e6,Thiago Moises,15,3,0,180.34,...,2.48,42,4.08,53,1.31,35,52,1.1,1,"September 02, 2023"


## Fill NA vals with column median

In [10]:
RelInfo = RelInfo.fillna(RelInfo.median(numeric_only=True))


## Save to CSV

In [ ]:
RelInfo.to_csv("data/processed/relInfo.csv")

# Fight history file without fighter stats:

FightHistory = RelInfo[["event_id", "r_name", "b_name", "winner_id", "date"]]
FightHistory.to_csv('data/processed/fighthistory.csv')

## Function to get name of winning fighter from any row (for later clarity)

In [187]:
def get_fighter(row):
    if row["winner_id"] == 1:
        return row["r_name"]
    else:
        return row["b_name"]

# Test: winner should be Ruffy     
win_fighter = get_fighter(RelCopy.iloc[1])
print(win_fighter)

Mauricio Ruffy


In [188]:
RelInfo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8337 entries, 0 to 8336
Data columns (total 36 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   event_id      8337 non-null   object 
 1   fight_id      8337 non-null   object 
 2   r_id          8337 non-null   object 
 3   r_name        8337 non-null   object 
 4   b_id          8337 non-null   object 
 5   b_name        8337 non-null   object 
 6   r_wins        8337 non-null   int64  
 7   r_losses      8337 non-null   int64  
 8   r_draws       8337 non-null   int64  
 9   r_height      8337 non-null   float64
 10  r_weight      8337 non-null   float64
 11  r_reach       8337 non-null   float64
 12  r_splm        8337 non-null   float64
 13  r_str_acc     8337 non-null   int64  
 14  r_sapm        8337 non-null   float64
 15  r_str_def     8337 non-null   int64  
 16  r_td_avg      8337 non-null   float64
 17  r_td_avg_acc  8337 non-null   int64  
 18  r_td_def      8337 non-null 

## Begin building model

## Random forest

In [191]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# 5 votes on 1 or 0, -> confidence score. ()
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

scaler = StandardScaler()
classifier = RandomForestClassifier(random_state=0)
# Features
X = RelInfo.drop(['event_id', 'fight_id', 'r_id','r_name', 'b_id', 'b_name', 'winner_id', 'date'], axis=1)
# label
y = RelInfo['winner_id']

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2,       
    random_state=42,     
    shuffle=True) 

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
classifier.fit(X_train_scaled, y_train)

predictions = classifier.predict(X_test_scaled)


accuracy = accuracy_score(predictions, y_test)
print(accuracy)




0.7110311750599521


# Logistic Regression

In [194]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(penalty='l2', random_state=42)
model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)
accuracy = accuracy_score(predictions, y_test)
print(accuracy)


0.7038369304556354


## Observations: 
Dataset leaks information the model should not have access to. The fighter stats are static and always reflect the latest statistics of each fighter (as late as September 2025), allowing the model to become more accurate due to this knowledge. Must have the stats of each fighter be accurate to WHEN fight took place. Possible solution: Scrape Wayback machine archives of different fighters to build a new fighter_details csv file.


## Resources: 
UFC contains most fighter stats on individual pages for each fighter e.i: https://www.ufc.com/athlete/khabib-nurmagomedov For Khabib Nurmagomedov

In [ ]:
import requests
from datetime import datetime
WAYBACK_URL = "https://web.archive.org/web"
UFC_URL = "https://www.ufc.com/athlete"

class UFCStatsScraper:
    def __init__(self, fighter_name, fight_date):
        self.fighter_name = fighter_name
        self.fight_date = fight_date
        self.TimeStamp = self.GetTimeStamp()

    def GetTimeStamp(self):
        TimeStamp = datetime.strptime(self.fight_date, "%B %d, %Y").strftime("%Y%m%d")
        return TimeStamp
    
    def GetSnapShot(self):
        SnapshotURL = f"{WAYBACK_URL}/{self.TimeStamp}/{UFC_URL}/{self.fighter_name}"
        return SnapshotURL
        

# Example
Scraper = UFCStatsScraper("Islam-Makhachev", "July 17, 2021")
Scraper.GetSnapShot()

'https://web.archive.org/web/20210717/https://www.ufc.com/athlete/Islam-Makhachev'